In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_customers = pd.read_csv('csv/olist_customers_dataset.csv')
df_geolocation = pd.read_csv('csv/olist_geolocation_dataset.csv')
df_order_items = pd.read_csv('csv/olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('csv/olist_order_payments_dataset.csv')
df_order_reviews = pd.read_csv('csv/olist_order_reviews_dataset.csv')
df_orders = pd.read_csv('csv/olist_orders_dataset.csv')
df_products = pd.read_csv('csv/olist_products_dataset.csv')
df_sellers = pd.read_csv('csv/olist_sellers_dataset.csv')
df_category_name_translation = pd.read_csv('csv/product_category_name_translation.csv')

Ana Omurga (Sipariş ve Ürün Akışı):

df_orders : Müşterinin sepeti onayladığı anı temsil eder. Siparişin ne zaman verildiği, onaylandığı, kargoya verildiği, müşteriye ulaştığı tarihler ve siparişin güncel durumu (teslim edildi, iptal edildi vb.) burada tutulur. Fiyat veya ürün bilgisi içermez.

df_order_items (Sipariş Detayları Tablosu): Bir siparişin (sepetin) içinde neler olduğunu gösterir. Ürünlerin fiyatı, kargo bedeli (freight_value), hangi satıcıdan alındığı ve ürünün sistemdeki benzersiz ID'si buradadır. Bir müşteri tek siparişte 3 ürün aldıysa, bu tabloda o sipariş ID'sine ait 3 satır olur.


Müşteri ve Satıcı Ağı:

df_customers (Müşteriler Tablosu): Siparişi veren kişilerin anonimleştirilmiş ID'leri, yaşadıkları şehir, eyalet ve posta kodları (zip code) bulunur. Her müşterinin benzersiz bir kimliği vardır.

df_sellers (Satıcılar Tablosu): Platform üzerinden satış yapan dükkanların bilgileri. Satıcının nerede (şehir/eyalet) bulunduğu bilgisini tutar.

df_geolocation (Coğrafi Konum Tablosu): Brezilya'daki posta kodlarının (zip code) enlem ve boylam (latitude/longitude) koordinatlarını tutar. Harita üzerinde yoğunluk grafiği çizmek istenirse kullanılır.


Ödeme ve Geri Bildirim:

df_order_payments (Ödemeler Tablosu): Müşterinin o siparişi nasıl ödediğini (kredi kartı, boleto/nakit, voucher/kupon), kaç taksit yaptığını ve toplam ödenen tutarı gösterir.

df_order_reviews (Yorumlar Tablosu): Sipariş tamamlandıktan sonra müşterinin verdiği 1 ile 5 arası puanlar ve (varsa) yazdığı yorum metinleri, yorum tarihleri buradadır.

Kritik Çeviri Dosyası:

df_category_name_translation (Kategori Çevirisi): Veri seti Brezilya'dan olduğu için df_products tablosundaki kategori isimleri Portekizcedir (örneğin: beleza_saude). Bu dosya, Portekizce isimleri İngilizceye (health_beauty) çeviren basit bir sözlüktür. LLM'in İngilizce veya Türkçe sorulan soruları doğru algılaması için bu çeviriyi ürünler tablosuna birleştirmek (merge/join) çok önemlidir.









In [3]:
list_of_dfs = [df_customers, df_geolocation, df_order_items, df_order_payments, df_order_reviews, df_orders, df_products, df_sellers, df_category_name_translation]

for df in list_of_dfs:
    print(len(df))
    print(df.isnull().sum())
    


99441
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
1000163
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
112650
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
103886
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
99224
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
99441
order_id                            0
custom

In [4]:
df_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [5]:
df_category_name_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [6]:
df_products_en = pd.merge(df_products, df_category_name_translation, on='product_category_name', how='left')
df_products_en = df_products_en.drop('product_category_name', axis=1)
df_products_en.head()

,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


# ***feature eng.***

In [7]:
# df_order_reviews
df_order_reviews['review_comment_title'] = df_order_reviews['review_comment_title'].fillna('No Title')
df_order_reviews['review_comment_message'] = df_order_reviews['review_comment_message'].fillna('No Comment')





In [8]:
df_products_en.isna().sum()

product_id                         0
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    623
dtype: int64

In [9]:
df_products_clean = df_products_en[['product_id', 'product_category_name_english']].copy()
df_products_clean['product_category_name_english'] = df_products_clean['product_category_name_english'].fillna('Unknown')
print(df_products_clean.isnull().sum())



product_id                       0
product_category_name_english    0
dtype: int64


In [10]:
list_of_dfs_updated = [df_customers, df_geolocation, df_order_items, df_order_payments, df_order_reviews, df_orders, df_products_clean, df_sellers]
for df in list_of_dfs_updated:
    print(df.isnull().sum())

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64
order_id                            0
customer_id                         0
order_status                        

In [11]:
import os

os.makedirs('cleaned_csv', exist_ok=True)

# 1. Customers (Müşteriler)
df_customers_clean = df_customers[['customer_id', 'customer_city', 'customer_state']]
df_customers_clean.to_csv('cleaned_csv/customers.csv', index=False)


# 2. Orders (Siparişler)
df_orders_clean = df_orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date']]
df_orders_clean.to_csv('cleaned_csv/orders.csv', index=False)

# 3. Order Items (Sipariş Detayları ve Fiyatlar)
df_order_items_clean = df_order_items[['order_id', 'order_item_id', 'product_id', 'price']]
df_order_items_clean.to_csv('cleaned_csv/order_items.csv', index=False)

# 4. Order Payments (Ödemeler)
df_order_payments_clean = df_order_payments[['order_id', 'payment_sequential', 'payment_type', 'payment_value']]
df_order_payments_clean.to_csv('cleaned_csv/payments.csv', index=False)

# 5. Order Reviews (Yorum Skorları)
df_order_reviews_clean = df_order_reviews[['review_id', 'order_id', 'review_score']]
df_order_reviews_clean.to_csv('cleaned_csv/reviews.csv', index=False)

# 6. Products (Daha önce hazırladığımız, sadece İngilizce Kategori ve ID olan tablo)
# (Eğer hafızada df_products_clean duruyorsa direkt çalıştır, yoksa yukarıdaki koddan tekrar oluştur)
df_products_clean.to_csv('cleaned_csv/products.csv', index=False)
